# Tarea 2 · Parte 1 — Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿Pueden convertirse documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Campo | Valor |
|---|---|
| Fecha de extracción | 2026-10-25 |
| Corpus A (documentos) | 25 imágenes/páginas de documentos peruanos de **Wikimedia Commons** (8 impresos, 8 fotos con sombra/inclinación, 9 manuscritos). Licencias: dominio público, CC BY-SA 3.0/4.0, CC BY 2.0 (detalle por archivo en `datos/fuentes.csv`). |
| Corpus B (audio) | 10 entrevistas reales en español peruano de **Wikimedia Commons** (CC BY-SA 4.0, registradas por estudiantes de la UNMSM) + 14 clips de **OpenSLR 73** (*Crowdsourced high-quality Peruvian Spanish*, CC BY-SA 4.0, Guevara-Rukoz et al., 2020). |
| OCR | PaddleOCR (PP-OCRv5 mobile: detector + reconocedor latino, `lang=es`, CPU) — versión impresa abajo |
| ASR | faster-whisper `small` (int8, CPU) + VAD Silero integrado; diarización con embeddings ECAPA (SpeechBrain) + clustering aglomerativo |
| Entornos | `uv sync` (equivalente a los grupos `docai` y `audio` de las sesiones) |

**Política de datos personales.** Se seudonimizan nombres de personas privadas vivas (entrevistados), DNI, teléfonos y correos.
No se seudonimizan figuras históricas públicas ni personas fallecidas hace más de un siglo presentes en documentos de archivo
(p. ej. registros parroquiales del s. XVII–XIX, Miguel Grau), porque la Ley 29733 protege datos de personas naturales vivas.
La tabla de correspondencias seudónimo↔nombre real se guarda solo en `datos/raw/` (no versionado).

In [1]:
import hashlib, json, time, warnings
from pathlib import Path
import torch  # importar torch antes que paddle evita un conflicto de DLL en Windows
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

import paddleocr, paddle, faster_whisper
import voz

RAIZ = Path(".")
RAW = RAIZ / "datos" / "raw"
CACHE = RAW / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
print("PaddleOCR", paddleocr.__version__, "| PaddlePaddle", paddle.__version__, "| faster-whisper", faster_whisper.__version__)

PaddleOCR 3.7.0 | PaddlePaddle 3.3.1 | faster-whisper 1.2.1


## 1. Corpus descargado
Los archivos se descargan con `datos.py` (API de Commons, User-Agent descriptivo, 1,5 s entre solicitudes; OpenSLR por HTTP Range sin bajar el zip completo de ~1 GB).
Los originales quedan en `datos/raw/` (ignorado por Git).

In [2]:
man_doc = pd.read_csv(RAW / "documentos" / "manifiesto.csv")
man_aud = pd.read_csv(RAW / "audio" / "manifiesto.csv")
print(man_doc.shape, man_aud.shape)
man_doc[["id", "tipo", "titulo", "pagina", "licencia"]]

(28, 10) (24, 10)


,id,tipo,titulo,pagina,licencia
0,doc_001,impreso,File:Constitución Política de la República Peruana (1828).pdf,5.0,Public domain
1,doc_002,impreso,File:Constitución Política de la República Peruana (1828).pdf,12.0,Public domain
2,doc_003,impreso,File:Constitución Política del Perú (1856).pdf,4.0,Public domain
3,doc_004,impreso,File:Constitución Política del Perú (1856).pdf,10.0,Public domain
4,doc_005,impreso,File:Constitución política del Perú reformada por el Congreso de 1860.pdf,5.0,Public domain
5,doc_006,impreso,File:Constitución política del Perú reformada por el Congreso de 1860.pdf,11.0,Public domain
6,doc_007,impreso,File:CODIGO DE ETICA PROFESIONAL DE LA PNP.pdf,6.0,CC0
7,doc_008,impreso,File:CODIGO DE ETICA PROFESIONAL DE LA PNP.pdf,2.0,CC0
8,doc_009,foto,File:Capitulación de Ayacucho1.jpg,NaN,Public domain
9,doc_010,foto,File:Capitulacón de Ayacucho2.jpg,NaN,Public domain


## 2. Limpieza: duplicados por hash y descartes

In [3]:
dup = man_doc[man_doc.duplicated("sha256", keep="first")]
print(f"Duplicados por SHA-256: {len(dup)} ->", dup[["id", "titulo"]].to_dict("records"))
man_doc = man_doc.drop_duplicates("sha256", keep="first").reset_index(drop=True)
dup_a = man_aud[man_aud.duplicated("sha256", keep="first")]
print(f"Audios duplicados: {len(dup_a)}")
man_aud = man_aud.drop_duplicates("sha256").reset_index(drop=True)
man_doc.tipo.value_counts()

Duplicados por SHA-256: 1 -> [{'id': 'doc_028', 'titulo': 'File:Estado General de Miguel Grau.jpg'}]
Audios duplicados: 0


tipo
foto          10
manuscrito     9
impreso        8
Name: count, dtype: int64

## 3. OCR con PaddleOCR
Parámetros: `lang="es"` (PP-OCRv5 mobile, reconocedor `latin` que cubre el español), sin orientación de documento ni *unwarping* (configuración base de la sesión 10),
imágenes reescaladas a 1280 px de ancho por Commons (o tamaño original si es menor). Se guarda el JSON crudo de cada documento en `datos/raw/documentos/ocr_paddle/`.

In [4]:
from paddleocr import PaddleOCR
ocr_dir = RAW / "documentos" / "ocr_paddle"; ocr_dir.mkdir(exist_ok=True)
ocr = PaddleOCR(lang="es", use_doc_orientation_classify=False, use_doc_unwarping=False,
                use_textline_orientation=False, enable_mkldnn=False,
                text_detection_model_name="PP-OCRv5_mobile_det",
                text_recognition_model_name="latin_PP-OCRv5_mobile_rec")

def ocr_paddle(ruta: str) -> tuple[str, float, float]:
    salida = ocr_dir / (Path(ruta).stem + ".json")
    if salida.exists():
        d = json.loads(salida.read_text(encoding="utf-8"))
        return d["texto"], d["conf_media"], d["segundos"]
    t0 = time.perf_counter()
    res = ocr.predict(ruta)[0]
    textos, confs = list(res["rec_texts"]), [float(c) for c in res["rec_scores"]]
    d = {"texto": "\n".join(textos), "conf_media": float(np.mean(confs)) if confs else 0.0,
         "segundos": time.perf_counter() - t0, "lineas": textos, "scores": confs,
         "cajas": [np.asarray(b).tolist() for b in res["rec_polys"]]}
    salida.write_text(json.dumps(d, ensure_ascii=False), encoding="utf-8")
    return d["texto"], d["conf_media"], d["segundos"]

filas = []
for r in man_doc.itertuples():
    texto, conf, seg = ocr_paddle(r.archivo)
    filas.append({"id": r.id, "texto_ocr_crudo": texto, "conf_ocr": round(conf, 3), "seg_ocr": round(seg, 1)})
    print(f"{r.id} {r.tipo:<10} conf={conf:.2f} chars={len(texto):>5} t={seg:.1f}s")
ocr_df = man_doc.merge(pd.DataFrame(filas), on="id")

Creating model: ('PP-OCRv5_mobile_det', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Karen\.paddlex\official_models\PP-OCRv5_mobile_det`.


Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Karen\.paddlex\official_models\latin_PP-OCRv5_mobile_rec`.


doc_001 impreso    conf=0.92 chars=  247 t=16.8s
doc_002 impreso    conf=0.95 chars= 1574 t=62.8s
doc_003 impreso    conf=0.41 chars=   15 t=4.2s
doc_004 impreso    conf=0.94 chars= 1324 t=61.0s
doc_005 impreso    conf=0.97 chars=  941 t=88.2s
doc_006 impreso    conf=0.98 chars= 1313 t=82.2s
doc_007 impreso    conf=0.98 chars= 2083 t=118.6s
doc_008 impreso    conf=0.93 chars= 1561 t=107.3s
doc_009 foto       conf=0.00 chars=    0 t=8.8s
doc_010 foto       conf=0.00 chars=    0 t=2.1s
doc_011 foto       conf=0.94 chars=  581 t=43.4s
doc_012 foto       conf=0.81 chars=  594 t=79.7s
doc_013 foto       conf=0.33 chars=   30 t=6.6s
doc_014 foto       conf=0.97 chars= 1490 t=117.5s
doc_015 foto       conf=0.98 chars=  446 t=41.5s
doc_016 foto       conf=0.79 chars=   35 t=14.9s
doc_017 foto       conf=0.96 chars=  181 t=36.3s
doc_018 foto       conf=0.75 chars=  101 t=15.7s
doc_019 manuscrito conf=0.44 chars=  277 t=73.0s
doc_020 manuscrito conf=0.86 chars=  350 t=42.1s
doc_021 manuscrito co

### Descartes de documentos ilegibles
Criterio fijado *antes* de ver métricas: el OCR marca como candidato todo documento con < 15 caracteres; se descarta solo si la
**inspección visual** confirma que no hay texto legible para una persona. Un fallo del OCR sobre un documento legible no es motivo de
descarte: ese documento queda como caso difícil.
Si algún documento se descarta, se reemplaza por uno del mismo tipo hasta cumplir 8/8/9.

In [5]:
candidatos = ocr_df[ocr_df.texto_ocr_crudo.str.len() < 15]
print("Candidatos (OCR < 15 caracteres):", candidatos[["id", "tipo", "titulo"]].to_dict("records") or "ninguno")
# Revisión visual de cada candidato (anotada a mano):
REVISION_VISUAL = {
    "doc_009": "ILEGIBLE: la imagen es una pintura de la firma de la Capitulación, no contiene texto -> se descarta",
    "doc_010": "legible: foto de baja resolución de la Capitulación de Ayacucho; se conserva como caso difícil",
}
for i in candidatos.id:
    print(f"  {i}: {REVISION_VISUAL.get(i, 'SIN REVISAR')}")
ilegibles = candidatos[candidatos.id.map(lambda i: REVISION_VISUAL.get(i, "ILEGIBLE")).str.startswith("ILEGIBLE")]
print("Descartados por ilegibles:", ilegibles.id.tolist() or "ninguno")
ocr_df = ocr_df.drop(ilegibles.index)
# Cuotas del enunciado; los documentos sobrantes de cada tipo son reservas que solo entran si se descarta otro
CUOTA = {"impreso": 8, "foto": 8, "manuscrito": 9}
reservas = pd.concat([g.iloc[CUOTA[t]:] for t, g in ocr_df.groupby("tipo")])
print("Reservas no usadas:", reservas[["id", "tipo", "titulo"]].to_dict("records"))
ocr_df = pd.concat([g.iloc[:CUOTA[t]] for t, g in ocr_df.groupby("tipo")]).sort_values("id").reset_index(drop=True)
man_doc = man_doc[man_doc.id.isin(ocr_df.id)]
print("Conteo final por tipo:", ocr_df.tipo.value_counts().to_dict(), "| total", len(ocr_df))

Candidatos (OCR < 15 caracteres): [{'id': 'doc_009', 'tipo': 'foto', 'titulo': 'File:Capitulación de Ayacucho1.jpg'}, {'id': 'doc_010', 'tipo': 'foto', 'titulo': 'File:Capitulacón de Ayacucho2.jpg'}]
  doc_009: ILEGIBLE: la imagen es una pintura de la firma de la Capitulación, no contiene texto -> se descarta
  doc_010: legible: foto de baja resolución de la Capitulación de Ayacucho; se conserva como caso difícil
Descartados por ilegibles: ['doc_009']
Reservas no usadas: [{'id': 'doc_018', 'tipo': 'foto', 'titulo': 'File:Peru revenue stamp on receipt.jpg'}]
Conteo final por tipo: {'manuscrito': 9, 'impreso': 8, 'foto': 8} | total 25


## 4. ASR: VAD + faster-whisper + diarización
- VAD Silero (`vad_filter=True`, silencios ≥ 500 ms) para evitar alucinaciones en silencios/ruido.
- `condition_on_previous_text=False` para cortar bucles de repetición; decodificación greedy (beam 1) por el costo en CPU de 4 núcleos.
- Diarización en **todas las entrevistas de Commons** (varias tienen entrevistador + entrevistado(s)); los clips de OpenSLR son de un solo hablante por construcción.
  Umbral de distancia coseno 0,70 (tope 4 hablantes). Un primer intento con 0,55 asignó el tope de 4 hablantes a *todas* las entrevistas
  (sobresegmentación evidente, incluso en entrevistas de una sola persona), por lo que se rehízo la diarización desde los SRT con
  `scripts/rediarizar.py` sin repetir el ASR. No hay referencia de hablantes, así que este umbral no se ajustó contra ninguna métrica.
- Exportación por clip a SRT y TXT en `datos/raw/audio/salidas/`.

In [6]:
sal_dir = RAW / "audio" / "salidas"; sal_dir.mkdir(exist_ok=True)

def procesar_audio(r) -> dict:
    cache = CACHE / f"asr_{r.id}.json"
    if cache.exists():
        return json.loads(cache.read_text(encoding="utf-8"))
    segs, meta = voz.transcribir(r.archivo, "small", vad=True)
    if r.tipo == "entrevista_commons" and segs:
        n_h = voz.diarizar(r.archivo, segs)
    else:
        n_h = 1
        for s in segs:
            s.hablante = None
    voz.exportar(segs, sal_dir / r.id)
    d = {"id": r.id, "duracion_seg": round(meta["duracion"], 1), "duracion_voz": round(meta["duracion_voz"], 1),
         "seg_proceso": round(meta["segundos_proceso"], 1), "idioma": meta["idioma"], "n_segmentos": len(segs),
         "texto_asr_crudo": voz.texto_con_hablantes(segs) if n_h > 1 else " ".join(s.texto for s in segs),
         "n_hablantes": n_h}
    cache.write_text(json.dumps(d, ensure_ascii=False), encoding="utf-8")
    return d

res = []
for r in man_aud.itertuples():
    d = procesar_audio(r)
    res.append(d)
    print(f"{r.id} {r.tipo:<18} {d['duracion_seg']:>6.1f}s voz={d['duracion_voz']:>6.1f}s hablantes={d['n_hablantes']} RTF={d['seg_proceso']/d['duracion_seg']:.2f}")
asr_df = man_aud.merge(pd.DataFrame(res), on="id")
print(f"Total audio: {asr_df.duracion_seg.sum()/60:.1f} min en {len(asr_df)} clips; multihablante: {(asr_df.n_hablantes>1).sum()} clips")

clip_001 entrevista_commons  136.3s voz= 114.2s hablantes=4 RTF=0.91
clip_002 entrevista_commons  189.0s voz= 170.4s hablantes=4 RTF=0.86
clip_003 entrevista_commons   29.9s voz=  28.4s hablantes=3 RTF=0.83
clip_004 entrevista_commons   54.3s voz=  51.7s hablantes=4 RTF=1.15
clip_005 entrevista_commons   95.0s voz=  79.7s hablantes=4 RTF=1.24
clip_006 entrevista_commons   77.4s voz=  74.5s hablantes=4 RTF=1.15
clip_007 entrevista_commons   98.3s voz=  12.0s hablantes=2 RTF=0.27
clip_008 entrevista_commons  250.6s voz= 247.6s hablantes=4 RTF=1.11
clip_009 entrevista_commons  196.0s voz= 187.8s hablantes=2 RTF=1.28
clip_010 entrevista_commons   96.1s voz=  89.7s hablantes=3 RTF=1.27
clip_011 openslr73            71.3s voz=  50.8s hablantes=1 RTF=0.78
clip_012 openslr73            75.9s voz=  45.0s hablantes=1 RTF=0.78
clip_013 openslr73            80.2s voz=  36.2s hablantes=1 RTF=0.58
clip_014 openslr73            66.6s voz=  39.0s hablantes=1 RTF=0.98
clip_015 openslr73            93.0

### Descarte de audios sin voz
Criterio: duración de voz tras VAD < 5 % de la duración total o transcripción vacía.

In [7]:
sin_voz = asr_df[(asr_df.duracion_voz < 0.05 * asr_df.duracion_seg) | (asr_df.texto_asr_crudo.str.strip() == "")]
print("Audios descartados por falta de voz:", sin_voz.id.tolist() or "ninguno")
asr_df = asr_df.drop(sin_voz.index).reset_index(drop=True)

Audios descartados por falta de voz: ninguno


## 5. Seudonimización
Nombres de personas privadas tomados de las descripciones de Commons de las entrevistas (no se imprimen aquí) + patrones DNI/teléfono/correo + nombres tras "me llamo / mi nombre es / señora…".
Después se hizo **revisión humana** de las transcripciones de las entrevistas leyendo los TXT. La revisión encontró (a) variantes de los
nombres deformadas por el ASR (apellidos mal transcritos, con *z* en lugar de *s*, pegados a otra palabra) y (b) nombres de
entrevistadores y entrevistados que no figuraban en las descripciones de Commons. Se añadieron a la lista privada (no versionada) y se
volvió a ejecutar esta celda. Nombres de futbolistas y figuras públicas mencionados en las entrevistas no se seudonimizan.

In [8]:
nombres_privados = json.loads((RAW / "nombres_privados.json").read_text(encoding="utf-8"))  # no versionado
tabla = {}
cont_doc = cont_aud = 0
ocr_df["texto_ocr"], n = zip(*[(t, k) for t, k, _ in (voz.seudonimizar(x, nombres_privados, tabla, disparadores=False) for x in ocr_df.texto_ocr_crudo)])
cont_doc = sum(n)
asr_df["texto_asr"], n = zip(*[(t, k) for t, k, _ in (voz.seudonimizar(x, nombres_privados, tabla) for x in asr_df.texto_asr_crudo)])
cont_aud = sum(n)
(RAW / "tabla_seudonimos.json").write_text(json.dumps(tabla, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"Reemplazos en documentos: {cont_doc} | en audio: {cont_aud} | entidades únicas: {len(tabla)}")
print("Clases de entidad reemplazadas:", pd.Series([v.split('_')[0].strip('[') for v in tabla.values()]).value_counts().to_dict())

Reemplazos en documentos: 1 | en audio: 30 | entidades únicas: 19
Clases de entidad reemplazadas: {'PERSONA': 19}


## 6. Análisis exploratorio (Plotly Express)

In [9]:
ocr_df["chars_ocr"] = ocr_df.texto_ocr.str.len()
asr_df["palabras_asr"] = asr_df.texto_asr.map(lambda t: len(voz.normalizar(t).split()))
asr_df["fuente_corta"] = asr_df.tipo.map({"entrevista_commons": "Entrevistas Commons", "openslr73": "OpenSLR 73"})

conteo = ocr_df.tipo.value_counts().rename_axis("tipo").reset_index(name="n")
conf_tipo = ocr_df.groupby("tipo").conf_ocr.mean().round(2).to_dict()
fig1 = px.bar(conteo, x="tipo", y="n", color="tipo", text="n",
              title=f"Corpus balanceado (8/8/9), pero la confianza OCR cae en manuscritos: {conf_tipo.get('manuscrito')} vs {conf_tipo.get('impreso')} en impresos",
              labels={"n": "documentos", "tipo": "tipo de documento"})
fig1.show()

In [10]:
fig2 = px.bar(asr_df.sort_values("duracion_seg"), x="id", y="duracion_seg", color="fuente_corta",
              title=f"{asr_df.duracion_seg.sum()/60:.0f} min de audio: las entrevistas de calle son más largas y variables que los clips de OpenSLR (~1 min)",
              labels={"duracion_seg": "duración (s)", "id": "clip", "fuente_corta": "fuente"})
fig2.show()

In [11]:
long_df = pd.concat([
    pd.DataFrame({"longitud": ocr_df.chars_ocr, "medida": "caracteres OCR (documentos)", "grupo": ocr_df.tipo}),
    pd.DataFrame({"longitud": asr_df.palabras_asr, "medida": "palabras ASR (audios)", "grupo": asr_df.fuente_corta}),
])
fig3 = px.histogram(long_df, x="longitud", color="grupo", facet_col="medida", nbins=20, barmode="overlay",
                    title="Los impresos concentran los textos OCR largos; los manuscritos casi no producen texto útil",
                    labels={"longitud": "longitud del texto"})
fig3.update_xaxes(matches=None)
fig3.show()

## 7. Validación: extremos de longitud

In [12]:
cols = ["id", "tipo", "chars_ocr", "conf_ocr"]
print("OCR más largos"); display(ocr_df.nlargest(3, "chars_ocr")[cols + ["texto_ocr"]].assign(texto_ocr=lambda d: d.texto_ocr.str[:150]))
print("OCR más cortos"); display(ocr_df.nsmallest(3, "chars_ocr")[cols + ["texto_ocr"]])
cols_a = ["id", "fuente_corta", "duracion_seg", "palabras_asr", "n_hablantes"]
print("Clips más largos"); display(asr_df.nlargest(3, "duracion_seg")[cols_a + ["texto_asr"]].assign(texto_asr=lambda d: d.texto_asr.str[:150]))
print("Clips más cortos"); display(asr_df.nsmallest(3, "duracion_seg")[cols_a + ["texto_asr"]].assign(texto_asr=lambda d: d.texto_asr.str[:150]))

OCR más largos


,id,tipo,chars_ocr,conf_ocr,texto_ocr
6,doc_007,impreso,2083,0.979,SECCION SEGUNDA\nCAPITULO II\nTITULO III\nDEBERES PARA CON LA SOCIEDAD\nDEBERES DEL POLICIA\nArt. 250.-\nRespetar y ...
1,doc_002,impreso,1574,0.955,"1*\n8\n \nART. 22. Tiene igualmente el deber de acusar\nante el senado al Presidente y Vice-Presidente, á bs\nmiembr..."
7,doc_008,impreso,1561,0.927,Vaw\nGCO690160030-0\nRUBEN ROMERO SANCHEZ\nTate Gral. PNP.\nIecot\nemerai do la Palia Noea\nResolución Suprema\ndel ...


OCR más cortos


,id,tipo,chars_ocr,conf_ocr,texto_ocr
8,doc_010,foto,0,0.000,
2,doc_003,impreso,15,0.410,TOM3TZ/0\nUNRTNO
11,doc_013,foto,30,0.332,HISTORIA DE HUACA LAS BALSAS\n\n


Clips más largos


,id,fuente_corta,duracion_seg,palabras_asr,n_hablantes,texto_asr
7,clip_008,Entrevistas Commons,250.6,631,4,"[HABLANTE_1] Bueno, hola, estamos en la San Martín, entre Fritas Patreas, 28 de julio. Entonces, primero podría come..."
8,clip_009,Entrevistas Commons,196.0,482,2,"[HABLANTE_1] Buenas noches, primeramente es mi nombre [PERSONA_16], soy estudiante de la Universidad San Marcos. A v..."
1,clip_002,Entrevistas Commons,189.0,350,4,"[HABLANTE_1] Buenos días profesor y compañeros, hoy estoy cerca de aquí por mi casa y voy a hacer una entrevista cer..."


Clips más cortos


,id,fuente_corta,duracion_seg,palabras_asr,n_hablantes,texto_asr
2,clip_003,Entrevistas Commons,29.9,63,3,"[HABLANTE_1] Normal, pero no resultaba nada. [HABLANTE_2] Pero como lo ves? Como ves a Perú ahorita? [HABLANTE_3] Co..."
3,clip_004,Entrevistas Commons,54.3,163,4,"[HABLANTE_1] Sí, dejólo ya. Así está la banda. [HABLANTE_2] ¿Qué ayuda? [HABLANTE_3] Ya está la banda. [HABLANTE_1] ..."
13,clip_014,OpenSLR 73,66.6,100,1,No quiero estar me mojando toda la tarde. La mermelada de sarsamora es muy fácil de hacer. Se han caído 13 edificios...


Coherencia con los histogramas: los OCR más largos son páginas impresas (cola derecha del histograma de caracteres) y los más cortos son manuscritos o fotos
con poco texto; los clips más largos son entrevistas de Commons y los más cortos, entrevistas breves de calle.

## 8. Exportación

In [13]:
DATOS = RAIZ / "datos"
documentos = ocr_df.assign(
    fuente=lambda d: "Wikimedia Commons (" + d.licencia + ") " + d.fuente_url,
    modelo_ocr=f"PaddleOCR {paddleocr.__version__} (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",
    texto_referencia="",
)[["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia", "conf_ocr", "chars_ocr"]]

transcripciones = asr_df.assign(
    idioma="es (español peruano, según la fuente)",
    modelo_asr="faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",
    texto_referencia="",
    fuente=lambda d: d.fuente_corta + " (" + d.licencia + ")",
)[["id", "archivo", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes", "texto_referencia", "fuente", "seg_proceso"]]
transcripciones = transcripciones.rename(columns={"archivo": "archivo_audio"})

fuentes = pd.concat([
    man_doc.assign(modalidad="documento", titulo_pagina=lambda d: [t if pd.isna(p) else f"{t} (p. {int(p)})" for t, p in zip(d.titulo, d.pagina)]),
    man_aud.assign(modalidad="audio", titulo_pagina=lambda d: d.titulo),
])[["id", "modalidad", "fuente_url", "licencia", "autor", "titulo_pagina"]]
fuentes["acceso"] = np.where(fuentes.fuente_url.str.contains("openslr"),
                             "HTTP Range sobre el zip de OpenSLR 73 con datos.py (hablante indicado en el título)",
                             "Descarga directa por la API de Commons con datos.py")
# Los títulos de algunas entrevistas incluyen nombres de entrevistados: se seudonimizan también aquí
fuentes["titulo_pagina"] = [voz.seudonimizar(t, nombres_privados, tabla, disparadores=False)[0] for t in fuentes.titulo_pagina]

documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)
fuentes.to_csv(DATOS / "fuentes.csv", index=False)
for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones), ("fuentes", fuentes)]:
    print(nombre, df.shape)
display(documentos.head())
display(transcripciones.head())

documentos (25, 9)
transcripciones (24, 10)
fuentes (49, 7)


,id,archivo,tipo,fuente,texto_ocr,modelo_ocr,texto_referencia,conf_ocr,chars_ocr
0,doc_001,datos/raw/documentos/doc_001.jpg,impreso,Wikimedia Commons (Public domain) https://commons.wikimedia.org/?curid=147419188,"CONSTITUCION POLITICA\nDE LA\nRZDHBLIEADERBARA,\nDADA POR\nEL CONGRESO GENERAL\nCONSTITUYENTE\nel dia 18 de Marzo de...","PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",,0.916,247
1,doc_002,datos/raw/documentos/doc_002.jpg,impreso,Wikimedia Commons (Public domain) https://commons.wikimedia.org/?curid=147419188,"1*\n8\n \nART. 22. Tiene igualmente el deber de acusar\nante el senado al Presidente y Vice-Presidente, á bs\nmiembr...","PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",,0.955,1574
2,doc_003,datos/raw/documentos/doc_003.jpg,impreso,Wikimedia Commons (Public domain) https://commons.wikimedia.org/?curid=147418775,TOM3TZ/0\nUNRTNO,"PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",,0.410,15
3,doc_004,datos/raw/documentos/doc_004.jpg,impreso,Wikimedia Commons (Public domain) https://commons.wikimedia.org/?curid=147418775,"3\nI.e Por sentencia en qué se imponga esa\npena, conforme á la ley:\n2.°\nPor quiebra fraudulenta ju licialmente\nd...","PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",,0.937,1324
4,doc_005,datos/raw/documentos/doc_005.jpg,impreso,Wikimedia Commons (Public domain) https://commons.wikimedia.org/?curid=147422002,"EL LIBERTADOR RAMON CASTILLA,\nPRESIDENTE CONSTITUCIONAL DE LA REPUBLICA, &C.\nPor cuanto el Congreso há sancionado ...","PaddleOCR 3.7.0 (PP-OCRv5 mobile det + latin rec, sin unwarping ni orientación)",,0.973,941


,id,archivo_audio,duracion_seg,idioma,texto_asr,modelo_asr,n_hablantes,texto_referencia,fuente,seg_proceso
0,clip_001,datos/raw/audio/clip_001.wav,136.3,"es (español peruano, según la fuente)","[HABLANTE_1] Buenos días, aquí son las 10 y 28 y estoy con una señora que va a darme su opinión acerca de partido de...","faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",4,,Entrevistas Commons (CC BY-SA 4.0),124.6
1,clip_002,datos/raw/audio/clip_002.wav,189.0,"es (español peruano, según la fuente)","[HABLANTE_1] Buenos días profesor y compañeros, hoy estoy cerca de aquí por mi casa y voy a hacer una entrevista cer...","faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",4,,Entrevistas Commons (CC BY-SA 4.0),163.4
2,clip_003,datos/raw/audio/clip_003.wav,29.9,"es (español peruano, según la fuente)","[HABLANTE_1] Normal, pero no resultaba nada. [HABLANTE_2] Pero como lo ves? Como ves a Perú ahorita? [HABLANTE_3] Co...","faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",3,,Entrevistas Commons (CC BY-SA 4.0),24.9
3,clip_004,datos/raw/audio/clip_004.wav,54.3,"es (español peruano, según la fuente)","[HABLANTE_1] Sí, dejólo ya. Así está la banda. [HABLANTE_2] ¿Qué ayuda? [HABLANTE_3] Ya está la banda. [HABLANTE_1] ...","faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",4,,Entrevistas Commons (CC BY-SA 4.0),62.2
4,clip_005,datos/raw/audio/clip_005.wav,95.0,"es (español peruano, según la fuente)","[HABLANTE_1] Ahora, mi nombre es [PERSONA_12], pero si eras no [PERSONA_11], ahora me voy a presentar a hacerle unas...","faster-whisper small int8 (VAD Silero on, greedy) + diarización ECAPA",4,,Entrevistas Commons (CC BY-SA 4.0),117.9
